In [ ]:
#@title Environment Setup


OPTIONS = {}

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = 'ComfyUI'
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

In [ ]:
#@title Install Nodes

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd /content/ComfyUI
!pip install -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/systemaiofinterest-wq/ComfyUI-Pixverse-AI-V2.git
%cd ComfyUI-Pixverse-AI-V2
!pip install -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/rgthree/rgthree-comfy.git
%cd rgthree-comfy
!pip install -r requirements.txt

%cd /content/ComfyUI/custom_nodes
#1. Clone the repo
!git clone https://github.com/yolain/ComfyUI-Easy-Use
%cd ComfyUI-Easy-Use
#2. Install the requirements
!Double-click install.bat to install the required dependencies

# Instalar nodo VideoHelperSuite en Google Colab

import os

CUSTOM_NODES_DIR = "/content/ComfyUI/custom_nodes"
NODE_DIR = os.path.join(CUSTOM_NODES_DIR, "ComfyUI-VideoHelperSuite")

# 1. Borrar carpeta si ya existe (para reinstalar/reemplazar)
if os.path.exists(NODE_DIR):
    print("🗑️ Borrando versión anterior...")
    !rm -rf "{NODE_DIR}"

# 2. Clonar repositorio
print("⬇️ Clonando ComfyUI-VideoHelperSuite...")
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git "{NODE_DIR}"

# 3. Instalar dependencias del nodo
print("📦 Instalando dependencias...")
%cd "{NODE_DIR}"
!pip install -r requirements.txt

# 4. Instalar ffmpeg (ES IMPORTANTE PARA VIDEO EN COLAB)
print("🎬 Instalando ffmpeg...")
!apt update -y
!apt install -y ffmpeg

# 5. Verificar instalación
print("\n✅ Verificando instalación...")
!ls -l

print(f"\n🎉 Nodo instalado en: {NODE_DIR}")

In [ ]:
# Descargar(checkpoint .safetensors)

%cd /content/drive/MyDrive/ComfyUI/models/checkpoints

!wget -O dreamshaper_8_pruned.safetensors \
https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors

In [ ]:
#@title Run ComfyUI with cloudflared (Recommended Way)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
%cd /content/drive/MyDrive/ComfyUI
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{port}",
        "--loglevel",
        "debug",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
  for line in p.stdout:
    print(line, end="")
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

device_conect = "GPU" #@param ["CPU", "GPU"]

if device_conect == "GPU":
  !python main.py --listen 0.0.0.0 --enable-cors-header "*" --dont-print-server
else:
  !python main.py --dont-print-server --cpu

In [ ]:
#@title Run ComfyUI with localtunnel
!npm install -g localtunnel
%cd /content/ComfyUI
import threading

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

device_conect = "GPU" #@param ["CPU", "GPU"]

if device_conect == "GPU":
  !python main.py --dont-print-server
else:
  !python main.py --dont-print-server --cpu